# Ali-CCP: Alibaba Click and Conversion Prediction (Colab)

This notebook: (1) bootstraps Colab (Drive, work dir, clone repo), (2) loads the [Ali-CCP dataset](https://tianchi.aliyun.com/dataset/408) from Tianchi, (3) preprocesses, trains an MLP for CVR/CTR (binary classification), and reports metrics.

**Data:** Download from https://tianchi.aliyun.com/dataset/408: `sample_train.tar.gz` (4.1GB) and `sample_test.tar.gz` (4.7GB). Upload both into DATA_DIR; the notebook will extract and parse them.

**Alternative:** Preprocessed CSV(s) from [torch-rechub](https://github.com/datawhalechina/torch-rechub/tree/main/examples/ranking/data/ali-ccp) (e.g. `ali_ccp_train.csv`, `ali_ccp_val.csv`, `ali_ccp_test.csv`) — if present, these take precedence over the archives.

In [ ]:
# Mount Google Drive for persistent storage
import os

if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
# Setup working directory in Google Drive
import os

if not os.path.exists('/content/drive'):
    print('Warning: Drive not mounted, some paths may not work.')
WORK_DIR = '/content/drive/MyDrive/colab/aliccp_dataset'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
# Clone repo, install dependencies, and make src importable (Colab-friendly)
import os, subprocess, shutil

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

# Gate for unattended papermill runs: when this notebook itself was synced into
# repo_dir via scp, rmtree+clone would delete the scp'd checkout (including this
# notebook) and replace it with a fresh GitHub clone, discarding local changes.
# Operator can set env var SKIP_GIT_REPO_SYNC=1 for scp+papermill runs.
SKIP_GIT_REPO_SYNC = os.environ.get('SKIP_GIT_REPO_SYNC', '0') == '1'

if IN_COLAB and not SKIP_GIT_REPO_SYNC:
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
    subprocess.run(['git', 'clone', repo_url], check=True)
    os.chdir(repo_dir)
    subprocess.run(['git', 'fetch', '--all'], check=True)
    subprocess.run(['git', 'checkout', branch_name], check=False)
elif os.path.isdir(repo_dir):
    os.chdir(repo_dir)
    print('SKIP_GIT_REPO_SYNC=True: using existing repo checkout, skipping clone/reset.')
else:
    print(f'SKIP_GIT_REPO_SYNC=True but {repo_dir!r} not found; staying in current directory.')

In [ ]:
# Install dependencies (omit jupyter - already in Colab)
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy', 'scikit-learn',
                'matplotlib', 'seaborn', 'requests', 'tqdm', 'joblib'], check=True)

In [ ]:
# Config
PROJECT_NAME = 'aliccp_dataset'
DATA_DIR = '/content/drive/MyDrive/colab/data/ali_ccp'
force_rewrite = False
TASK_TYPE = 'classification'
TARGET_COL = 'purchase'  # CVR; use 'click' for CTR
SAMPLE_SIZE = 500_000  # Use 500k rows for Colab; set None for full
RANDOM_STATE = 42

In [ ]:
# Data acquisition (manual: user must download from Tianchi and place files in DATA_DIR)
import os
import tarfile
import pandas as pd
from tqdm import tqdm

os.makedirs(DATA_DIR, exist_ok=True)

# Tianchi archive names (from Data List page)
SAMPLE_TRAIN_TAR = 'sample_train.tar'
SAMPLE_TEST_TAR = 'sample_test.tar'

# Preprocessed CSV names (from torch-rechub or similar)
TRAIN_CSV = 'ali_ccp_train.csv'
VAL_CSV = 'ali_ccp_val.csv'
TEST_CSV = 'ali_ccp_test.csv'
SINGLE_CSV = 'ali_ccp.csv'  # alternative: one file with all data

# Raw Tianchi file names (inside extracted archives)
SAMPLE_SKELETON_TRAIN = 'sample_skeleton_train.csv'
SAMPLE_SKELETON_TEST = 'sample_skeleton_test.csv'
COMMON_FEATURES_TRAIN = 'common_features_train.csv'
COMMON_FEATURES_TEST = 'common_features_test.csv'

SPARSE_COLS = ['101', '121', '122', '124', '125', '126', '127', '128', '129', '205', '206', '207', '210', '216', '508', '509', '702', '853', '301', '109_14', '110_14', '127_14', '150_14']
DENSE_COLS = ['109_14', '110_14', '127_14', '150_14', '508', '509', '702', '853']
USES_COLS = SPARSE_COLS + ['D' + c for c in DENSE_COLS]

def _find_file_recursive(root, filename):
    """Find filename under root (direct or in subdirs)."""
    direct = os.path.join(root, filename)
    if os.path.isfile(direct):
        return direct
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

def _parse_feat_str(feat_str, sparse_cols, dense_cols):
    feat_dict = {}
    for fstr in feat_str.split('\x01'):
        if '\x02' not in fstr or '\x03' not in fstr:
            continue
        parts = fstr.split('\x02', 1)
        filed = parts[0]
        feat_val = parts[1]
        if '\x03' in feat_val:
            feat, val = feat_val.split('\x03', 1)
            if filed in sparse_cols:
                feat_dict[filed] = feat
            if filed in dense_cols:
                feat_dict['D' + filed] = val
    return feat_dict

def parse_raw_ali_ccp(data_dir, sample_size=None):
    """Parse raw Tianchi files with memory-efficient two-pass approach.

    Pass 1: scan skeleton files (limited by sample_size) to collect needed
            common-feature IDs instead of loading all ~20 GB into RAM.
    Pass 2: load only the referenced common features.
    Pass 3: re-read skeleton rows and merge with common features.
    """
    common_train_path = _find_file_recursive(data_dir, COMMON_FEATURES_TRAIN)
    common_test_path = _find_file_recursive(data_dir, COMMON_FEATURES_TEST)
    skeleton_train_path = _find_file_recursive(data_dir, SAMPLE_SKELETON_TRAIN)
    skeleton_test_path = _find_file_recursive(data_dir, SAMPLE_SKELETON_TEST)
    if not all([common_train_path, common_test_path, skeleton_train_path, skeleton_test_path]):
        return None, None

    test_limit = (sample_size // 4) if sample_size else None

    # Pass 1: collect common-feature IDs referenced by the skeleton rows we will use
    needed_ids = set()
    for path, limit, mode in [
        (skeleton_train_path, sample_size, 'train'),
        (skeleton_test_path, test_limit, 'test'),
    ]:
        with open(path, 'r') as f:
            for i, line in enumerate(tqdm(f, desc=f'scan_skeleton_{mode}', leave=False)):
                if limit and i >= limit:
                    break
                parts = line.strip().split(',')
                if len(parts) >= 4:
                    needed_ids.add(parts[3])
    print(f'Unique common-feature IDs needed: {len(needed_ids):,}')

    # Pass 2: load only the needed common features
    common_feat = {}
    for path, mode in [(common_train_path, 'train'), (common_test_path, 'test')]:
        with open(path, 'r') as f:
            for line in tqdm(f, desc=f'common_features_{mode}', leave=False):
                parts = line.strip().split(',')
                if len(parts) < 3:
                    continue
                if parts[0] not in needed_ids:
                    continue
                feat_dict = _parse_feat_str(parts[2], SPARSE_COLS, DENSE_COLS)
                common_feat[parts[0]] = feat_dict
    print(f'Loaded {len(common_feat):,} common-feature entries')

    # Pass 3: parse skeleton rows and merge with common features
    rows_train, rows_test = [], []
    for path, rows_out, limit, mode in [
        (skeleton_train_path, rows_train, sample_size, 'train'),
        (skeleton_test_path, rows_test, test_limit, 'test'),
    ]:
        with open(path, 'r') as f:
            for i, line in enumerate(tqdm(f, desc=f'sample_skeleton_{mode}', leave=False)):
                if limit and i >= limit:
                    break
                parts = line.strip().split(',')
                if len(parts) < 6:
                    continue
                click, purchase = parts[1], parts[2]
                if click == '0' and purchase == '1':
                    continue
                feat_dict = _parse_feat_str(parts[5], SPARSE_COLS, DENSE_COLS)
                feat_dict.update(common_feat.get(parts[3], {}))
                row = {'click': click, 'purchase': purchase}
                for k in USES_COLS:
                    row[k] = feat_dict.get(k, '0')
                rows_out.append(row)
    df_train = pd.DataFrame(rows_train)
    df_test = pd.DataFrame(rows_test)
    return df_train, df_test

train_path = os.path.join(DATA_DIR, TRAIN_CSV)
val_path = os.path.join(DATA_DIR, VAL_CSV)
test_path = os.path.join(DATA_DIR, TEST_CSV)
single_path = os.path.join(DATA_DIR, SINGLE_CSV)
train_tar_path = os.path.join(DATA_DIR, SAMPLE_TRAIN_TAR)
test_tar_path = os.path.join(DATA_DIR, SAMPLE_TEST_TAR)

# Extract archives if present and CSVs missing (or force_rewrite)
has_archives = os.path.isfile(train_tar_path) and os.path.isfile(test_tar_path)
needs_extract = has_archives and (
    force_rewrite or
    not _find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) or
    not _find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN)
)
if needs_extract:
    for arc in [SAMPLE_TRAIN_TAR, SAMPLE_TEST_TAR]:
        path = os.path.join(DATA_DIR, arc)
        if os.path.isfile(path):
            print(f'Extracting {arc}...')
            with tarfile.open(path, 'r:*') as tf:
                tf.extractall(DATA_DIR)
            print(f'Done.')

has_splits = os.path.exists(train_path) and os.path.exists(test_path)
has_single = os.path.exists(single_path)
has_raw = (
    _find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) is not None and
    _find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN) is not None
)

if has_splits:
    print(f'Using preprocessed splits: {train_path}, {test_path}')
    if os.path.exists(val_path):
        print(f'Validation: {val_path}')
elif has_single:
    print(f'Using single CSV: {single_path}')
elif has_raw:
    print('Parsing raw Tianchi files (sample_skeleton_*.csv, common_features_*.csv)...')
    df_train, df_test = parse_raw_ali_ccp(DATA_DIR, sample_size=SAMPLE_SIZE)
    if df_train is not None and len(df_train) > 0:
        n_test = (SAMPLE_SIZE // 4) if SAMPLE_SIZE else len(df_test)
        df_test = df_test.head(n_test) if n_test else df_test
        print(f'Parsed: train {len(df_train)}, test {len(df_test)}')
    else:
        has_raw = False
        df_train = df_test = None

if not has_splits and not has_single and not has_raw:
    print('No data found in DATA_DIR. Please:')
    print('  1. Go to https://tianchi.aliyun.com/dataset/408')
    print('  2. Sign in and download: sample_train.tar.gz, sample_test.tar.gz')
    print(f'  3. Upload both files into: {DATA_DIR}')
    print('  4. Re-run this cell. The notebook will extract the archives and parse the data.')

In [ ]:
# Inspect
import pandas as pd

if has_splits:
    df_train = pd.read_csv(train_path, nrows=SAMPLE_SIZE)
    df_test = pd.read_csv(test_path, nrows=(SAMPLE_SIZE // 4) if SAMPLE_SIZE else None)
    print('Train shape:', df_train.shape)
    print('Test shape:', df_test.shape)
    print('Train dtypes:')
    print(df_train.dtypes)
    print('\nTrain head:')
    print(df_train.head())
    display(df_train.head())
    df = df_train  # for compatibility with single-file flow
elif has_single:
    df = pd.read_csv(single_path, nrows=SAMPLE_SIZE)
    print('Shape:', df.shape)
    print('\nDtypes:')
    print(df.dtypes)
    print('\nHead:')
    print(df.head())
    display(df.head())
elif has_raw and df_train is not None:
    print('Train shape:', df_train.shape)
    print('Test shape:', df_test.shape)
    print('Train dtypes:')
    print(df_train.dtypes)
    print('\nTrain head:')
    print(df_train.head())
    display(df_train.head())
    df = df_train
else:
    raise FileNotFoundError('Run the Data acquisition cell and add data to DATA_DIR first.')

In [ ]:
# Task: binary classification (CVR or CTR)
task = TASK_TYPE
print(f'Task: {task}')
print(f'Target: {TARGET_COL}')
print('Metrics: Accuracy, F1, AUC')

In [ ]:
# Preprocess
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

def get_feature_columns(df, target_col):
    exclude = {'click', 'purchase', 'split'}
    return [c for c in df.columns if c not in exclude]

def safe_transform(le, ser):
    """Transform series; unseen categories -> '__NA__' then encode."""
    s = ser.astype(str).fillna('__NA__')
    known = set(le.classes_)
    s = s.map(lambda x: x if x in known else '__NA__')
    return le.transform(s)

if has_splits or has_raw:
    feature_cols = get_feature_columns(df_train, TARGET_COL)
    X_train = df_train[feature_cols].copy()
    y_train = df_train[TARGET_COL].astype(int).values
    X_test = df_test[feature_cols].copy()
    y_test = df_test[TARGET_COL].astype(int).values
    if os.path.exists(val_path):
        df_val = pd.read_csv(val_path, nrows=SAMPLE_SIZE // 4 if SAMPLE_SIZE else None)
        X_val = df_val[feature_cols].copy()
        y_val = df_val[TARGET_COL].astype(int).values
    else:
        X_train, X_val, y_train, y_val = train_test_split(
            X_train, y_train, test_size=0.1, random_state=RANDOM_STATE, stratify=y_train
        )
        if len(X_val) == 0:
            X_val, y_val = None, None
else:
    feature_cols = get_feature_columns(df, TARGET_COL)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[TARGET_COL])
    X_train = train_df[feature_cols].copy()
    y_train = train_df[TARGET_COL].astype(int).values
    X_test = test_df[feature_cols].copy()
    y_test = test_df[TARGET_COL].astype(int).values
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=RANDOM_STATE, stratify=y_train
    )
    if len(X_val) == 0:
        X_val, y_val = None, None

# Encode categorical (object) columns; unseen in test/val -> '__NA__'
for c in feature_cols:
    if X_train[c].dtype == object or pd.api.types.is_object_dtype(X_train[c]):
        le = LabelEncoder()
        train_vals = X_train[c].astype(str).fillna('__NA__')
        le.fit(np.append(train_vals.values, ['__NA__']))
        X_train[c] = le.transform(train_vals)
        X_test[c] = safe_transform(le, X_test[c])
        if X_val is not None:
            X_val[c] = safe_transform(le, X_val[c])

# All to float and scale
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
if X_val is not None:
    X_val = X_val.astype(np.float32)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
if X_val is not None:
    X_val = scaler.transform(X_val)

n_features = X_train.shape[1]
print(f'Features: {n_features}, Train: {len(y_train)}, Test: {len(y_test)}')
if y_val is not None:
    print(f'Val: {len(y_val)}')

In [ ]:
# Model + train
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class MLPClassifier(nn.Module):
    def __init__(self, n_features, hidden_dims=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLPClassifier(n_features).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

X_t = torch.FloatTensor(X_train)
y_t = torch.FloatTensor(y_train).unsqueeze(1)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=1024, shuffle=True)

epochs = 15
losses = []
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb.squeeze(-1))
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    print(f'Epoch {epoch+1}/{epochs} loss={avg:.4f}')

print('Training done.')

In [ ]:
# Report
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# Training curve
plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Training Curve')
plt.grid(True)
plt.tight_layout()
plt.show()

# Test metrics
model.eval()
with torch.no_grad():
    X_te_t = torch.FloatTensor(X_test).to(device)
    logits = model(X_te_t).cpu().numpy()
    pred_proba = 1 / (1 + np.exp(-logits))
    pred = (pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, zero_division=0)
auc = roc_auc_score(y_test, pred_proba) if len(np.unique(y_test)) > 1 else 0.0

print('Test Metrics:')
print(f'  Accuracy: {acc:.4f}')
print(f'  F1: {f1:.4f}')
print(f'  AUC: {auc:.4f}')
print('\nConfusion matrix:')
print(confusion_matrix(y_test, pred))